# Bài 35: Huấn luyện một CNN (Training a CNN)

CNN ở Bài 34 được huấn luyện và đánh giá trên dữ liệu lấy từ cùng một phân phối khá rộng rãi. Huấn luyện thực tế có một kiểu thất bại sắc bén hơn nhiều đang rình rập: với quá ít dữ liệu và quá nhiều năng lực mô hình, một mạng có thể ghi nhớ hoàn hảo tập huấn luyện của nó trong khi không học được gì có thể tổng quát hóa. Bài 32 đã cho thấy nhiều năng lực hơn nghĩa là mô hình *có thể* biểu diễn nhiều hơn; bài này cho thấy mặt trái — chính năng lực dư thừa đó, với quá ít dữ liệu để ràng buộc nó, lại chính là thứ cho phép một mạng ghi nhớ thay vì tổng quát hóa. Bài này làm cho kiểu thất bại đó trở nên cụ thể, rồi khắc phục nó theo ba cách khác nhau: **tăng cường dữ liệu (data augmentation)** (các phép biến đổi ở Bài 8, được dùng vào mục đích mới), **chính quy hóa (regularization)** (suy giảm trọng số — weight decay), và **dropout**.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một tập dữ liệu nhỏ, khó có chủ đích

Cùng nhiệm vụ dấu-cộng-và-hình-tròn như Bài 34, nhưng giờ chỉ có **12 ảnh huấn luyện** và nhiễu điểm ảnh được thêm vào mọi ảnh, trong khi tập kiểm định (validation) vẫn lớn (150 ảnh) để độ chính xác của nó là một ước lượng đáng tin cậy.

In [ ]:
def make_image(shape_type, cx, cy, size=16, rng=None):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy - 1:cy + 2, cx - 3:cx + 4] = 1.0
        img[cy - 3:cy + 4, cx - 1:cx + 2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx - cx) ** 2 + (yy - cy) ** 2) <= 9] = 1.0
    if rng is not None:
        img = np.clip(img + rng.normal(0, 0.4, img.shape), 0, 1).astype(np.float32)
    return img

def make_dataset(rng_local, n, position_range):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy, rng=rng_local))
        labels.append(0.0 if shape_type == 'plus' else 1.0)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.float32)

data_rng = np.random.default_rng(2)
X_train, y_train = make_dataset(data_rng, 12, (3, 13))
X_val, y_val = make_dataset(data_rng, 150, (3, 13))

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_train[:6]):
    ax.imshow(im, cmap='gray')
    ax.axis('off')
fig.suptitle('The entire training set is only 12 noisy images like these', y=1.05)
plt.show()

## Quan sát nó quá khớp

Huấn luyện một CNN khá lớn (kiến trúc ở Bài 34) chỉ trên 12 ảnh này, và theo dõi cả hàm mất mát huấn luyện lẫn hàm mất mát *kiểm định* (trên 150 ảnh được giữ lại) ở mọi epoch.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1)).squeeze(-1)

def train_tracked(model_cls, Xtr, ytr, Xval, yval, epochs=400, lr=0.01, weight_decay=0.0, seed=0):
    torch.manual_seed(seed)  # seed BEFORE constructing the model, so init is actually reproducible
    model = model_cls()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr), ytr)
        loss.backward()
        opt.step()
        with torch.no_grad():
            train_losses.append(loss.item())
            val_losses.append(F.binary_cross_entropy_with_logits(model(Xval), yval).item())
    with torch.no_grad():
        train_acc = ((model(Xtr) > 0).float() == ytr).float().mean().item()
        val_acc = ((model(Xval) > 0).float() == yval).float().mean().item()
    return train_losses, val_losses, train_acc, val_acc

Xtr_t = torch.tensor(X_train).unsqueeze(1); ytr_t = torch.tensor(y_train)
Xval_t = torch.tensor(X_val).unsqueeze(1); yval_t = torch.tensor(y_val)

train_losses, val_losses, train_acc, val_acc = train_tracked(CNN, Xtr_t, ytr_t, Xval_t, yval_t)

print(f'final train accuracy: {train_acc:.1%}')
print(f'final val accuracy:   {val_acc:.1%}')
print(f'val loss minimum was {min(val_losses):.3f} at epoch {np.argmin(val_losses)} '
      f'(out of {len(val_losses)}); it ended at {val_losses[-1]:.3f}')

plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.axvline(np.argmin(val_losses), color='gray', linestyle='--', linewidth=1, label='best val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(fontsize=8)
plt.title('The classic overfitting curve')
plt.show()

Hàm mất mát huấn luyện đều đặn tiến về 0 &mdash; mạng ghi nhớ hoàn hảo cả 12 ảnh, kể cả nhiễu. Trong khi đó, hàm mất mát kiểm định chạm đáy ở giữa quá trình huấn luyện rồi leo ngược trở lại: qua điểm đó, mỗi epoch thêm vào lại khiến mô hình sai *một cách tự tin hơn* về dữ liệu mà nó chưa thấy. Độ chính xác kiểm định cuối cùng thấp hơn nhiều so với điểm số hoàn hảo trên tập huấn luyện, dù tập huấn luyện được khớp chính xác.

Quan sát diễn biến qua từng epoch giúp minh họa khái niệm: hàm mất mát kiểm định đi xuống theo hàm mất mát huấn luyện một thời gian, rồi tách ra thấy rõ và bắt đầu leo ngược trở lại ngay khoảng epoch 277, ngay cả khi hàm mất mát huấn luyện tiếp tục đi xuống thấp hơn.

<img src="../animations/overfitting_demo.gif" alt="Ảnh động các đường cong hàm mất mát huấn luyện và kiểm định của một CNN quá khớp tập dữ liệu 12 ảnh: hàm mất mát huấn luyện giảm đều về gần 0 trong khi hàm mất mát kiểm định vẫn nhiễu rồi leo ngược trở lại sau epoch 277, với một đường nét đứt đánh dấu hàm mất mát kiểm định tốt nhất khi huấn luyện đạt tới nó" style="display: block; margin: 0 auto; max-width: 700px;">

## Cách khắc phục 1: tăng cường dữ liệu

Nếu không có đủ dữ liệu thật, hãy sản xuất thêm từ những gì đang có. Áp dụng các phép biến đổi ngẫu nhiên ở Bài 8 (lật, xoay nhẹ) lên mỗi ảnh huấn luyện &mdash; nhãn không thay đổi, nhưng các điểm ảnh thì có, nên mạng thấy được sự đa dạng rộng hơn nhiều về "một dấu cộng/hình tròn có thể trông như thế nào" thay vì ghi nhớ 12 ảnh chính xác.

In [ ]:
def augment(img, rng_local):
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    if rng_local.random() < 0.5:
        img = np.flipud(img).copy()
    angle = rng_local.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((8, 8), angle, 1.0)
    img = cv2.warpAffine(img, M, (16, 16))
    return img.astype(np.float32)

aug_rng = np.random.default_rng(5)
X_aug, y_aug = [], []
for _ in range(20):  # 20 augmented copies of each of the 12 original images
    for img, label in zip(X_train, y_train):
        X_aug.append(augment(img, aug_rng))
        y_aug.append(label)
X_aug, y_aug = np.array(X_aug, dtype=np.float32), np.array(y_aug, dtype=np.float32)

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_aug[:6]):
    ax.imshow(im, cmap='gray')
    ax.axis('off')
fig.suptitle(f'6 of {len(X_aug)} augmented copies, all still "the same 12 base images"', y=1.05)
plt.show()

Xaug_t = torch.tensor(X_aug).unsqueeze(1); yaug_t = torch.tensor(y_aug)
_, _, aug_train_acc, aug_val_acc = train_tracked(CNN, Xaug_t, yaug_t, Xval_t, yval_t, epochs=150)
print(f'with augmentation: train acc = {aug_train_acc:.1%}, val acc = {aug_val_acc:.1%}  (was {val_acc:.1%})')

Ảnh động bên dưới áp dụng `augment` lên một ảnh huấn luyện duy nhất lặp đi lặp lại, cho thấy những phép biến đổi nào được dùng mỗi lần.

<img src="../animations/augmentation_gallery_demo.gif" alt="Ảnh động tăng cường dữ liệu: một ảnh huấn luyện duy nhất luân phiên qua vài biến thể được lật và xoay ngẫu nhiên, với các phép biến đổi được áp dụng có nhãn và nhãn lớp giữ cố định suốt quá trình" style="display: block; margin: 0 auto; max-width: 480px;">

Cùng ý tưởng đó, áp dụng lên một bức ảnh thật (chỉ để minh họa &mdash; không thuộc lần huấn luyện của bài này). Các quy trình thực tế thường tận dụng một túi mẹo rộng hơn: không chỉ lật và xoay mà còn dao động độ sáng/tương phản/màu sắc, nhiễu Gauss, làm mờ, che khuất ngẫu nhiên, v.v. &mdash; các tổ hợp khác nhau, nhưng luôn là cùng một chú chó.

<img src="../animations/dog_augmentation_demo.gif" alt="Ảnh động tăng cường dữ liệu áp dụng lên ảnh chụp thật một chú chó: bức ảnh luân phiên qua các biến thể kết hợp lật, xoay ngẫu nhiên, dao động độ sáng/tương phản/màu sắc, nhiễu Gauss, làm mờ và che khuất, có nhãn cho biết phép biến đổi nào được áp dụng, mà vẫn rõ ràng là cùng một chú chó mỗi lần" style="display: block; margin: 0 auto; max-width: 480px;">

<p class="image-attribution">Ảnh của <a href="https://unsplash.com/@laulaco?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Laula Co</a> trên <a href="https://unsplash.com/photos/long-coat-tan-puppy-sICtHYZzt_s?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Unsplash</a></p>

## Cách khắc phục 2: suy giảm trọng số

**Suy giảm trọng số (weight decay)** thêm một mức phạt tỉ lệ với bình phương độ lớn trọng số trực tiếp vào hàm mất mát (tương đương, nó thu nhỏ mọi trọng số một chút về phía 0 ở mỗi lần cập nhật). Các trọng số lớn, được tinh chỉnh kỹ chính là thứ một mạng cần để ghi nhớ 12 ảnh nhiễu cụ thể; phạt độ lớn trọng số làm cho việc ghi nhớ đó tốn kém hơn so với việc tìm ra một hàm đơn giản hơn, mượt hơn &mdash; mà không cần thêm một mẫu huấn luyện nào.

In [ ]:
_, _, wd_train_acc, wd_val_acc = train_tracked(CNN, Xtr_t, ytr_t, Xval_t, yval_t, weight_decay=0.05)
print(f'with weight_decay=0.05: train acc = {wd_train_acc:.1%}, val acc = {wd_val_acc:.1%}  (was {val_acc:.1%})')

print()
print(f'{"approach":>20} {"train acc":>12} {"val acc":>12}')
print(f'{"no fix":>20} {train_acc:>12.1%} {val_acc:>12.1%}')
print(f'{"augmentation":>20} {aug_train_acc:>12.1%} {aug_val_acc:>12.1%}')
print(f'{"weight decay":>20} {wd_train_acc:>12.1%} {wd_val_acc:>12.1%}')

Cả hai cách khắc phục đều lấy lại được một phần lớn độ chính xác kiểm định đã mất, từ hai góc độ khác nhau: tăng cường dữ liệu tấn công vấn đề bằng cách cho mô hình nhiều dữ liệu (tổng hợp) hơn để làm đúng; suy giảm trọng số tấn công bằng cách làm cho mô hình ngay từ đầu ít sẵn lòng uốn mình quanh một tập dữ liệu nhỏ hơn. Trên thực tế, cả hai thường được dùng cùng nhau, cùng với các bộ chính quy hóa khác như **dropout**, được trình bày tiếp theo, và **chuẩn hóa theo lô (batch normalization)** (Bài 37), vốn tình cờ cũng hoạt động như một bộ chính quy hóa nhẹ.

## Cách khắc phục 3: dropout

**Dropout** (<a href="../references.html#srivastava-2014">Srivastava et al., 2014</a><span class="landmark-paper">&#9733;</span>) ngẫu nhiên đưa về 0 một tỉ lệ `p` các giá trị kích hoạt của một lớp ở mỗi bước huấn luyện, buộc các đơn vị còn sống sót không được dựa vào việc một đơn vị cụ thể nào khác luôn có mặt. Để giữ đầu ra của lớp ở cùng một tỉ lệ tổng thể bất kể dropout có đang hoạt động hay không, các giá trị kích hoạt sống sót được co giãn lại theo `1 / (1 - p)` — đây là "dropout đảo ngược (inverted dropout)", thứ mà lớp `Dropout` của mọi framework thực sự cài đặt. Khi đánh giá, dropout hoàn toàn không làm gì: lớp đầy đủ, không sửa đổi được chạy, đó là lý do `model.eval()` (đã được dùng xuyên suốt bài này, cho cả suy giảm trọng số và tăng cường dữ liệu) quan trọng — quên nó sẽ khiến dropout tiếp tục kích hoạt ngẫu nhiên lúc kiểm tra.

In [ ]:
def manual_dropout(x, p, rng_gen):
    keep_prob = 1 - p
    mask = (torch.rand(x.shape, generator=rng_gen) < keep_prob).float()
    return x * mask / keep_prob  # rescale so E[output] == input

x_demo = torch.randn(2000, 10)
torch.manual_seed(0)
out_torch = F.dropout(x_demo, p=0.3, training=True)
out_manual = manual_dropout(x_demo, 0.3, torch.Generator().manual_seed(0))

print(f'fraction zeroed, torch:  {(out_torch == 0).float().mean().item():.3f}  (target p = 0.3)')
print(f'fraction zeroed, manual: {(out_manual == 0).float().mean().item():.3f}')
print(f'mean before dropout: {x_demo.mean().item():.4f}')
print(f'mean after dropout (torch):  {out_torch.mean().item():.4f}  (rescaling keeps this close to the input mean)')
print(f'mean after dropout (manual): {out_manual.mean().item():.4f}')

eval_mode_out = F.dropout(x_demo, p=0.3, training=False)
print(f'eval-mode dropout is a no-op: {torch.equal(eval_mode_out, x_demo)}')

Ảnh động bên dưới áp dụng `manual_dropout` lên một lưới nhỏ cố định các giá trị kích hoạt, với một mặt nạ ngẫu nhiên mới ở mỗi bước: mỗi lần khoảng $\sim$ 30% đơn vị khác nhau bị loại bỏ, các đơn vị sống sót được co giãn lại theo $1/(1-p)$.

<img src="../animations/dropout_demo.gif" alt="Ảnh động dropout: một lưới các giá trị kích hoạt có 30 phần trăm đơn vị ngẫu nhiên khác nhau bị đưa về 0 ở mỗi bước, với các đơn vị sống sót được co giãn lại theo 1 chia (1 trừ p)" style="display: block; margin: 0 auto; max-width: 700px;">

Giờ hãy áp dụng nó vào bài toán quá khớp của bài này. Dropout cần *một chút* dư thừa để làm việc — đưa về 0 một nửa của một vector đặc trưng 32 đơn vị đi thẳng vào một đầu ra 1 đơn vị để lại rất ít dư địa để giúp ích, nên hãy thêm một lớp ẩn rộng hơn (32 → 64 → 1) và đặt dropout trên lớp 64 đơn vị. Với chỉ 12 ảnh huấn luyện, kết quả dao động nhiều giữa các seed ngẫu nhiên, nên hãy so sánh độ chính xác kiểm định trung bình qua vài seed thay vì tin vào một lần chạy duy nhất.

In [ ]:
class CNNDropout(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc1 = nn.Linear(32, 64)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        feat = torch.relu(self.fc1(self.conv(x).flatten(1)))
        return self.fc2(self.dropout(feat)).squeeze(-1)

def val_acc_for(dropout_p, seed, epochs=150):
    torch.manual_seed(seed)
    model = CNNDropout(dropout_p)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr_t), ytr_t)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        return ((model(Xval_t) > 0).float() == yval_t).float().mean().item()

no_drop_accs = [val_acc_for(0.0, seed) for seed in range(8)]
drop_accs = [val_acc_for(0.5, seed) for seed in range(8)]

print(f'no dropout:     mean val acc = {np.mean(no_drop_accs):.1%}  (+/- {np.std(no_drop_accs):.1%}, 8 seeds)')
print(f'dropout(0.5):   mean val acc = {np.mean(drop_accs):.1%}  (+/- {np.std(drop_accs):.1%}, 8 seeds)')

Trong một thiết lập đồ chơi nhỏ bé như thế này (chỉ 12 ảnh huấn luyện), chưa có nhiều dư thừa để dropout khai thác. Dù ở đây chỉ cho thấy một cải thiện khiêm tốn, lợi ích của dropout đã được khẳng định rõ ở quy mô của các tập dữ liệu thật và mạng thật (hàng trăm đơn vị dư thừa, hàng nghìn mẫu). Trên thực tế, dropout hầu như luôn được kết hợp với các bộ chính quy hóa khác trong trang này, không được dùng đơn lẻ.

### So sánh cả ba cách khắc phục

Mỗi cách khắc phục từ đầu đến giờ đều được thử riêng lẻ so với đường cơ sở. Ảnh động bên dưới so sánh cả bốn đường cong độ chính xác kiểm định cùng nhau trên cùng tập dữ liệu 12 ảnh: đường cơ sở không chính quy hóa kẹt ở mức thấp và nhiễu, tăng cường dữ liệu leo nhanh nhất và ổn định ở mức cao nhất, suy giảm trọng số mạnh nhưng biến động, và dropout &mdash; như đã nêu &mdash; chỉ cải thiện khiêm tốn so với đường cơ sở trên một tập dữ liệu nhỏ bé như vậy.

<img src="../animations/regularizer_race_demo.gif" alt="Ảnh động cuộc đua của bốn đường cong độ chính xác kiểm định qua 200 epoch: đường cơ sở không chính quy hóa kẹt quanh 60 phần trăm và nhiễu, tăng cường dữ liệu leo nhanh nhất và ổn định trên 95 phần trăm, suy giảm trọng số đạt khoảng 80 phần trăm nhưng biến động, và dropout ở gần đường cơ sở nhiễu" style="display: block; margin: 0 auto; max-width: 760px;">

## Ghép lại: ảnh thật, quá khớp thật

Bài toán đồ chơi 12 ảnh tổng hợp ở trên được chọn đặc biệt để làm cho kiểu thất bại trở nên không thể nhầm lẫn. Mục kết thúc này lặp lại thí nghiệm trên ảnh chụp thật **CIFAR-10** (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>): một tập huấn luyện nhỏ, khan hiếm dữ liệu có chủ đích gồm mèo và xe tải, với tăng cường dữ liệu, suy giảm trọng số và dropout được kết hợp cùng nhau. Ảnh chụp thật chứa ánh sáng, sự lộn xộn và sự mơ hồ thị giác thực sự giữa các lớp.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

ensure_cifar10()
cifar_train_imgs, cifar_train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    cifar_train_imgs.append(imgs)
    cifar_train_labels.append(labels)
cifar_train_imgs = np.concatenate(cifar_train_imgs)
cifar_train_labels = np.concatenate(cifar_train_labels)
cifar_test_imgs, cifar_test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take_class(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CIFAR_LABELS.index(name))[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx].copy()

cifar_rng = np.random.default_rng(4)
N_CIFAR_TRAIN = 40  # per class -- deliberately small, real training data is expensive
X_cifar_train = np.concatenate([
    take_class(cifar_train_imgs, cifar_train_labels, 'cat', N_CIFAR_TRAIN, cifar_rng),
    take_class(cifar_train_imgs, cifar_train_labels, 'truck', N_CIFAR_TRAIN, cifar_rng),
])
y_cifar_train = np.array([0.0] * N_CIFAR_TRAIN + [1.0] * N_CIFAR_TRAIN, dtype=np.float32)
X_cifar_val = np.concatenate([
    take_class(cifar_test_imgs, cifar_test_labels, 'cat', 150, cifar_rng),
    take_class(cifar_test_imgs, cifar_test_labels, 'truck', 150, cifar_rng),
])
y_cifar_val = np.array([0.0] * 150 + [1.0] * 150, dtype=np.float32)

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im in zip(axes, X_cifar_train[:6]):
    ax.imshow(im)
    ax.axis('off')
fig.suptitle(f'6 of {len(X_cifar_train)} real training photos (cat vs. truck)', y=1.05)
plt.show()

Cùng công thức như trước, được điều chỉnh cho đầu vào 3 kênh: một CNN đơn thuần không chính quy hóa, so với cùng kiến trúc đó với tăng cường dữ liệu, suy giảm trọng số và dropout đều được áp dụng cùng nhau trong một lần chạy duy nhất. Mỗi cấu hình được lấy trung bình qua 3 seed, vì 80 ảnh huấn luyện vẫn đủ nhỏ để các lần chạy riêng lẻ dao động.

In [ ]:
class CNNRGB(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )
        self.fc1 = nn.Linear(32, 64)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        feat = torch.relu(self.fc1(self.conv(x).flatten(1)))
        return self.fc2(self.dropout(feat)).squeeze(-1)

def augment_photo(img, rng_local):
    # unlike `augment` above, no vertical flip: upside-down cats and trucks aren't realistic
    if rng_local.random() < 0.5:
        img = np.fliplr(img).copy()
    angle = rng_local.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((16, 16), angle, 1.0)
    img = cv2.warpAffine(img, M, (32, 32))
    return img.astype(np.float32)

Xcv_t = torch.tensor(X_cifar_val).permute(0, 3, 1, 2); ycv_t = torch.tensor(y_cifar_val)

def train_cifar(use_aug, weight_decay, dropout_p, seed, epochs=300, lr=0.001):
    torch.manual_seed(seed)
    if use_aug:
        local_aug_rng = np.random.default_rng(seed + 100)
        Xa, ya = [], []
        for _ in range(10):  # 10 augmented copies of each of the 80 base images
            for img, label in zip(X_cifar_train, y_cifar_train):
                Xa.append(augment_photo(img, local_aug_rng))
                ya.append(label)
        Xtr_np, ytr_np = np.array(Xa, dtype=np.float32), np.array(ya, dtype=np.float32)
    else:
        Xtr_np, ytr_np = X_cifar_train, y_cifar_train
    Xct_t = torch.tensor(Xtr_np).permute(0, 3, 1, 2); yct_t = torch.tensor(ytr_np)
    model = CNNRGB(dropout_p=dropout_p)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xct_t), yct_t)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        train_acc = ((model(Xct_t) > 0).float() == yct_t).float().mean().item()
        val_acc = ((model(Xcv_t) > 0).float() == ycv_t).float().mean().item()
    return train_acc, val_acc

cifar_configs = [
    ('no fix',       dict(use_aug=False, weight_decay=0.0,  dropout_p=0.0)),
    ('augmentation', dict(use_aug=True,  weight_decay=0.0,  dropout_p=0.0)),
    ('weight decay', dict(use_aug=False, weight_decay=0.01, dropout_p=0.0)),
    ('dropout',      dict(use_aug=False, weight_decay=0.0,  dropout_p=0.3)),
    ('all combined', dict(use_aug=True,  weight_decay=0.01, dropout_p=0.3)),
]

print(f'{"approach":>15} {"train acc":>12} {"val acc":>12}')
cifar_results = {}
for name, cfg in cifar_configs:
    tr_accs, val_accs = [], []
    for seed in range(3):
        tr, va = train_cifar(seed=seed, **cfg)
        tr_accs.append(tr)
        val_accs.append(va)
    cifar_results[name] = (np.mean(tr_accs), np.mean(val_accs))
    print(f'{name:>15} {np.mean(tr_accs):>12.1%} {np.mean(val_accs):>12.1%}')

Không có cách khắc phục đơn lẻ nào là người thắng rõ ràng ở đây. Riêng suy giảm trọng số thực ra còn cho kết quả *thấp hơn* đường cơ sở không chính quy hóa — với `weight_decay=0.01`, độ chính xác huấn luyện thậm chí không còn đạt 100%, nghĩa là mức phạt đủ mạnh để chống lại mục tiêu mà không tìm ra được nghiệm tốt hơn thay thế. Đó là một kết quả hữu ích, khiến ta khiêm tốn: giá trị `weight_decay=0.05` dùng ở phần trước của bài được tinh chỉnh cho một mạng nhỏ hơn nhiều trên 12 ảnh tổng hợp nhỏ xíu, và một cường độ chính quy hóa hoạt động tốt trong một thiết lập không tự động chuyển giao sang một kiến trúc khác hay dữ liệu ảnh chụp thật.

Tăng cường dữ liệu và dropout mỗi cái riêng lẻ lấy lại được một lượng khiêm tốn độ chính xác kiểm định.

"Kết hợp tất cả" vượt qua mọi cách khắc phục đơn lẻ và đường cơ sở không chính quy hóa, lặp lại đúng bài học từ tập dữ liệu tổng hợp: ba bộ chính quy hóa này cộng dồn tác dụng chứ không thay thế cho nhau. Mức cải thiện so với đường cơ sở không chính quy hóa là có thật nhưng khiêm tốn (vài điểm phần trăm ở đây), và bản thân điều đó là bài học trung thực: chính quy hóa thu hẹp khoảng cách giữa hiệu năng huấn luyện và kiểm định, nó không tạo ra dữ liệu hay năng lực mô hình vốn không có.

Thu hẹp khoảng cách đó hơn nữa, trên ảnh thật, đòi hỏi những gì các bài sau giới thiệu — nhiều dữ liệu huấn luyện hơn, kiến trúc tốt hơn (Bài 37), và cuối cùng là **học chuyển giao (transfer learning)** (Bài 38): bắt đầu từ một mạng đã được huấn luyện trên hàng triệu ảnh, thay vì ghi nhớ từ đầu trên một nhúm ảnh.

### Bài tập

1. Thử các giá trị `weight_decay` là `0.001`, `0.05` và `1.0`. Có điểm nào mà nó bắt đầu *gây hại* cho độ chính xác huấn luyện cùng với (cuối cùng là) độ chính xác kiểm định không? Một mức suy giảm trọng số quá lớn làm gì với năng lực khớp bất cứ thứ gì của mô hình?
2. Quét `dropout_p` qua `[0.0, 0.2, 0.4, 0.6, 0.8]` trong `val_acc_for`, lấy trung bình qua cùng 8 seed ở mỗi giá trị. Có giá trị nào rõ ràng là tốt nhất không, hay giá trị trung bình nằm trong phạm vi một độ lệch chuẩn trên hầu hết khoảng giá trị vì có quá ít dữ liệu?
3. Tăng hệ số tăng cường từ 20 lên 100 bản sao cho mỗi ảnh gốc. Độ chính xác kiểm định có tiếp tục cải thiện, hay chững lại — và nếu chững lại, điều đó gợi ý gì về giới hạn cơ bản của việc tăng cường một tập dữ liệu ngay từ đầu chỉ chứa 12 mẫu *gốc*?